# Tutorial 2: Phase analysis with tree search
Dara is equipped with a parallelized tree search algorithm to identify possible phases
present in a given XRD pattern.

In this tutorial, we will try to identify the phases in one experimental solid-state
reaction sample between `GeO2` and `ZnO`.

> You can download this tutorial project from [here](https://idocx.github.io/dara/_static/tutorial.zip).

In [ ]:
%pip install ipywidgets nbformat

/Users/radical-rhys/Radical/dara/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path

from dara import search_phases

In [ ]:
pattern_path = "tutorial_data/GeO2-ZnO_700C_60min.xrdml"

# three elements are present in the sample
chemical_system = "Ge-O-Zn"

## Step 1: Prepare reference phases

Dara pre-builds an index of all the unique and low-energy phases in ICSD and COD
databases. It also implements a method to download CIF structures from COD data server
so that there is no need to obtain the offline database.

Before every search, we will need to gather all the reference phases in the chemical
system for the search algorithm. Dara provides `ICSDDatabase` and `CODDatabase` to do
the filtering.

In this example, we will use `CODDatabase` to download all the phases in the chemical system of `Ge-O-Zn`.

In [ ]:
from dara.structure_db import CODDatabase

# The COD database contains methods to filter phases in the chemical system
cod_database = CODDatabase()

# gather reference phases and save them to a directory called "cifs"
all_icsd_ids = cod_database.get_cifs_by_chemsys(chemical_system, dest_dir="cifs")

2026-02-18 15:16:49,128 WARNING dara.structure_db Local copy of database not found. Attempting to download structures...


2026-02-18 15:16:53,074 INFO dara.structure_db Saving downloaded CIFs to dara_downloaded_cifs
Skipping high-energy phase: 1528389 (Ge, 96): e_hull = 0.1494
Skipping high-energy phase: 9013109 (Ge, 64): e_hull = 0.3137
2026-02-18 15:16:53,084 INFO dara.structure_db Skipping common gas: O2
2026-02-18 15:16:53,084 INFO dara.structure_db Skipping common gas: O2
2026-02-18 15:16:53,085 INFO dara.structure_db Skipping common gas: O2
2026-02-18 15:16:53,085 INFO dara.structure_db Skipping common gas: O2
2026-02-18 15:16:53,085 INFO dara.structure_db Skipping common gas: O2
2026-02-18 15:16:53,085 INFO dara.structure_db Skipping common gas: O2
2026-02-18 15:16:53,086 INFO dara.structure_db Skipping common gas: O2
2026-02-18 15:16:53,086 INFO dara.structure_db Skipping common gas: O2
2026-02-18 15:16:53,086 INFO dara.structure_db Skipping common gas: O2
2026-02-18 15:16:53,086 INFO dara.structure_db Skipping common gas: O2
2026-02-18 15:16:53,086 INFO dara.structure_db Skipping common gas: O2
2

Since we are using a pre-filterd database (i.e., the COD), the downloaded CIF files will automatically be named according to the
following convention:

```
{composition}_{spacegroup}_(cod|icsd_{id})-{e_hull}.cif
```
Where the `e_hull` is the energy above the convex hull in meV/atom, as determined from
the Materials Project database for the ground-state entry with matching composition and spacegroup.

## Step 2: Search for phases

After preparing the reference CIFs, we can start the phase search on a provided XRD
pattern.

In this case, we are using the XRD pattern from the solid-state reaction sample
on our laboratory's Aeris diffractometer (`tutorial_data/GeO2-ZnO_700C_60min.xrdml`).

In [ ]:
# gather all the phases in the "cifs" directory
all_cifs = list(Path("cifs").glob("*.cif"))

search_results = search_phases(
    pattern_path=pattern_path,
    phases=all_cifs,
    instrument_profile="Aeris-fds-Pixcel1d-Medipix3",
)

2026-02-18 15:16:54,636	INFO worker.py:1927 -- Started a local Ray instance.


2026-02-18 15:16:55,071 INFO dara.search.tree Detecting peaks in the pattern.
2026-02-18 15:17:11,466 INFO dara.search.tree The wmax is automatically adjusted to 60.04.
2026-02-18 15:17:11,467 INFO dara.search.tree The intensity threshold is automatically set to 9.06 % of maximum peak intensity.
2026-02-18 15:17:11,467 INFO dara.search.tree Creating the root node.
2026-02-18 15:17:11,467 INFO dara.search.tree Refining all the phases in the dataset.
2026-02-18 15:17:32,636 INFO dara.search.tree The initial value of eps2 is automatically set to 0.000000_-0.05^0.05.
2026-02-18 15:17:32,638 INFO dara.search.tree Finished refining 92 phases, with 34 phases removed.
2026-02-18 15:17:32,638 INFO dara.search.tree Express mode is enabled. Grouping phases before starting.
2026-02-18 15:17:33,270 INFO dara.search.tree Phases are grouped into 45 groups. In express mode, only the best phase in each group will be considered during the search.
(_remote_expand_node pid=14431) 2026-02-18 15:17:33,311 I

## Step 3: Result analysis
The returned search result will be a list of `SearchResult` object.

In [ ]:
search_results

[SearchResult(refinement_result=RefinementResult(lst_data=LstResult(raw_lst='Rietveld refinement to file(s) GeO2-ZnO_700C_60min.xy\nBGMN version 4.2.23, 4614 measured points, 135 peaks, 24 parameters\nStart: Wed Feb 18 15:17:37 2026; End: Wed Feb 18 15:17:38 2026\n21 iteration steps\n\nRp=9.83%  Rpb=18.72%  R=10.35%  Rwp=12.11% Rexp=2.68%\nDurbin-Watson d=0.10\n1-rho=2.04%\n\nGlobal parameters and GOALs\n****************************\nQGeO2152cod23003650=0.4809+-0.0021\nQZnO186cod90041780=0.3862+-0.0024\nQZn2GeO4148cod90146310=0.1329+-0.0013\nEPS2=-0.002894+-0.000012\n\nLocal parameters and GOALs for phase GeO2152cod23003650\n******************************************************\nSpacegroupNo=152\nHermannMauguin=P3_121\nXrayDensity=4.276\nRphase=11.17%\nUNIT=NM\nA=0.499118+-0.000020\nC=0.564812+-0.000033\nk1=0.0100000\nB1=0.00500000\nGEWICHT=0.2613+-0.0011\nGrainSize(1,1,1)=84.1811\nAtomic positions for phase GeO2152cod23003650\n---------------------------------------------\n  3     0.

In this pattern, we only have one solution found with `Rwp = 12.04 %`.

In [ ]:
for i in range(len(search_results)):
    print(f"Rwp of solution {i} = {search_results[i].refinement_result.lst_data.rwp} %")

Rwp of solution 0 = 12.11 %


Each `SearchResult` has a `.visualize()` method to visualize the refined pattern and
missing/extra peaks in the solution. If there are no missing or extra peaks, this option
will not appear.

In [ ]:
search_results[0].visualize()

You can also view all the alternative phases in one solution from `SearchResult.phases` attribute.

In [ ]:
print("Phases found in solution 0:")
for i, phases_ in enumerate(search_results[0].phases):
    print(f"    - Phase {i}: {[phase.path.name for phase in phases_]}")

Phases found in solution 0:
    - Phase 0: ['GeO2_152_(cod_2300365)-0.cif', 'GeO2_154_(cod_9007477)-0.cif']
    - Phase 1: ['ZnO_186_(cod_9004178)-0.cif']
    - Phase 2: ['Zn2GeO4_148_(cod_9014631)-0.cif']


From the result, you can see that for the phase `GeO2`, the algorithm identifies two
similar phases with slightly different spacegroups (152 and 154).